In [18]:
import numpy as np
import pandas as pd
import joblib
import os

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
plt.style.use('src/plot_publication.styles.txt')
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Trajectory information -------------------------------------------------------

traj_length_ns = 1000  # Number of ns of the trajectory (1 microsecond)
chunk_length_ns = 25  # Each chunk is 100 ns long
dt = 5 # 5 ps 

chunks_start_ns = np.arange(0, traj_length_ns, chunk_length_ns).astype(int)
chunks_lst = [f'{i}_{i + chunk_length_ns}' for i in chunks_start_ns]

MOL = 'water_chloride'
DATA_DIR = '/media/ledoux/MYBOOK/waterchannels_LEDOUX/awc-honeycombs/helix_H5/crystal_plus_membrane'
DATA_DIR_WAT = f'/data/ledoux/awc-honeycombs/helix_H5/permeations/water'
DATA_DIR_CL = f'/data/ledoux/awc-honeycombs/helix_H5/permeations/chloride'
SYSTEM = 'H5cm'
FIG = 'fig/interactions'
topology = f'{DATA_DIR}/crystal_membrane_solv_ions.pdb'
TRAJ_DIR = f'{DATA_DIR}/processed'

PORES_IDX = {
    1: '1-10332',
    2: '10333-20664',
    3: '20665-30996',
    4: '30997-41328',
    5: '41329-51660',
    6: '51661-61992', 
    7: '61993-72324',
    'MESOPORE': '1-72324'
}

In [9]:
def process_permeations(data_dir, chunks_lst, offset):
    perms_lst = []

    for chunk in chunks_lst:
        chunk_start, _ = [int(t) for t in chunk.split('_')]

        file = f'{data_dir}/permeation_{SYSTEM}_{chunk}.joblib'

        try:
            df = joblib.load(file)
        except:
            print(f'cant read {file}')
            continue
        df = df.iloc[::offset, :].reset_index()
        df['chunk_entry_frame'] = df['end_frame'] - df['duration_nframe']
        df['chunk_entry_time_ns'] = (df['chunk_entry_frame'] * df['PS_PER_FRAME']) / 1000 # Convert from ps to ns
        df['chunk_exit_time_ns'] = (df['end_frame'] * df['PS_PER_FRAME']) / 1000 # Convert from ps to ns
        df['duration_ns'] = (df['duration_nframe'] * df['PS_PER_FRAME']) / 1000
        df['traj_entry_time_ns'] = df['chunk_entry_time_ns'] + chunk_start
        df['traj_exit_time_ns'] = df['chunk_exit_time_ns'] + chunk_start
        df['chunk_start_time'] = chunk_start

        perms_lst.append(df)

    perms = pd.concat(perms_lst, axis=0).reset_index(drop=True)
    
    return perms

In [ ]:
# Create scripts for processing on baal

mesopore_sel = "{not resname POPC NA CL SOL}"
within_membrane_sel = "z > 40 and z < 80"
dist_cutoff = 3.6
ang_cutoff = 30
outdir = "hbonds"

topology = f'processed/crystal_membrane_solv_ions.gro'
TRAJ_DIR = f'processed'

# ----------------------------------------------------------------------------

# Create output dir for the chunk

for chunk in chunks_lst:

    chunk_outdir = f"{outdir}/{chunk}"
    try:
        os.mkdir(chunk_outdir)
    except:
        pass

    # Read permeations
    perms_wat = process_permeations(
        DATA_DIR_WAT, [chunk], offset=10)  # Read perms every 10 perms

    # VMD selection and arguments for the chunk
    # All water that permeate in the chunk
    perms_sel = "{" + \
        f"resname SOL and resid {' '.join([str(w) for w in perms_wat['resid']])} and {within_membrane_sel}" + "}"
    # Path to trajectory
    trajectory = f'{TRAJ_DIR}/{SYSTEM}_{chunk}_processed.xtc'

    # ----------------------------------------------------------------------------
    # Write TCL script for VMD to calculate Hbonds between each permeating water and :
    # the mesopore or the other permeating waters

    with open(f"run_hbonds_{chunk}.tcl", "w") as fout:
        fout.write(f"""#!tcl

source src/hbonds_detailed.tcl

mol new     {topology}
mol addfile {trajectory} waitfor all

set sel1 [atomselect 0 {mesopore_sel}]
set sel3 [atomselect 0 {perms_sel}]

    # ----------------------------------------------------------------------------
    """)

        for w, water in perms_wat.iterrows():
            # Atom selection string
            water_sel = "{" + \
                f"resname SOL and resid {water['resid']} and {within_membrane_sel}" + "}"
            # Frames range of permeation
            entry_frame, exit_frame = int(
                water['start_frame']), int(water['end_frame'])
            # Output name for mesopore/permeating water hbonds <w>_<water resindex>_<water resid>_mesopore
            prefixout_meso_wat = f"{w}_{water['resindex']}_{water['resid']}_mesopore"
            # Output name for permeating water/water hbonds <w>_<water resindex>_<water resid>_water
            prefixout_wat_wat = f"{w}_{water['resindex']}_{water['resid']}_water"

            # TCL command for calculation
            fout.write(f"""set sel2 [atomselect 0 {water_sel}]
hbonds_detailed -sel1 $sel1 -sel2 $sel2 \
-dist {dist_cutoff} -ang {ang_cutoff} -polar yes -DA both -type all \
-frames {entry_frame}:{exit_frame} -outdir {chunk_outdir} \
-outfile {prefixout_meso_wat}.details.dat \
-detailout {prefixout_meso_wat}.stats.dat \
-writefile yes

hbonds_detailed -sel1 $sel3 -sel2 $sel2 \
-dist {dist_cutoff} -ang {ang_cutoff} -polar yes -DA both -type all \
-frames {entry_frame}:{exit_frame} -outdir {chunk_outdir} \
-outfile {prefixout_wat_wat}.details.dat \
-detailout {prefixout_wat_wat}.stats.dat \
-writefile yes
$sel2 delete

    # ----------------------------------------------------------------------------
    """)

        fout.write("quit\n\n")